In [ ]:
!pip install ultralytics kagglehub opencv-python

In [ ]:
import kagglehub

# Download dataset
path = kagglehub.dataset_download("awsaf49/bdd100k-dataset")

print("Dataset path:", path)

In [ ]:
import os

for root, dirs, files in os.walk(path):
    print(root)
    break

In [ ]:
from ultralytics import YOLO
import cv2
import os

# Load pretrained YOLO model
model = YOLO("yolov8n.pt")

# Example: pick any image from dataset
image_path = path + "/images/100k/train/0000f77c-6257be58.jpg"  # change if needed

# Read image
img = cv2.imread(image_path)

# Run detection
results = model(img)

# Show result
annotated = results[0].plot()

from google.colab.patches import cv2_imshow
cv2_imshow(annotated)

In [ ]:
import os

print("Root path:", path)
print(os.listdir(path))

In [ ]:
import glob

# Find all jpg images inside dataset
image_files = glob.glob(path + "/**/*.jpg", recursive=True)

print("Total images found:", len(image_files))
print("Sample images:", image_files[:5])

In [ ]:
import cv2

image_path = image_files[0]  # pick first image
print("Using:", image_path)

img = cv2.imread(image_path)

if img is None:
    print("❌ Failed again")
else:
    print("✅ Image loaded")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model(img)

In [ ]:
h, w, _ = img.shape

for r in results:
    for box in r.boxes:
        cls = int(box.cls[0])
        label = model.names[cls]
        x1, y1, x2, y2 = box.xyxy[0]

        if x1 > w * 0.7:
            print(f"⚠️ ALERT: {label} in RIGHT blind spot")

        if x2 < w * 0.3:
            print(f"⚠️ ALERT: {label} in LEFT blind spot")

In [ ]:
import glob
import cv2
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

# Load model
model = YOLO("yolov8n.pt")

# Get all images
image_files = glob.glob(path + "/**/*.jpg", recursive=True)

print("Total images:", len(image_files))

# Test on first 5 images
for img_path in image_files[:5]:
    print("\nProcessing:", img_path)

    img = cv2.imread(img_path)

    if img is None:
        continue

    results = model(img)
    annotated = results[0].plot()

    cv2_imshow(annotated)

In [ ]:
for img_path in image_files[:5]:
    img = cv2.imread(img_path)
    if img is None:
        continue

    results = model(img)

    h, w, _ = img.shape

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            label = model.names[cls]
            conf = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Draw bounding box
            cv2.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(img, f"{label} {conf:.2f}", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

            # RIGHT blind spot
            if x1 > w * 0.7:
                cv2.putText(img, "⚠ RIGHT BLIND SPOT!", (50,50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 3)

            # LEFT blind spot
            if x2 < w * 0.3:
                cv2.putText(img, "⚠ LEFT BLIND SPOT!", (50,100),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 3)

    cv2_imshow(img)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
video_file_name = list(uploaded.keys())[0]
cap = cv2.VideoCapture(video_file_name)

frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    results = model(frame)

    # your detection code here...

    if frame_count % 20 == 0:   # show every 20th frame
        cv2_imshow(frame)

In [ ]:
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output.mp4', fourcc, 20.0, (640,480))

# inside loop:
out.write(frame)

# after loop:
out.release()